# Vite、npm 与前端构建

## 4.1 之后还剩哪些问题

模块化解决了脚本顺序和全局污染，但开发体验仍然不够好：

| 问题 | 具体表现 |
|------|------|
| 看效果太麻烦 | 改一行代码就要 push、服务器 pull、再刷新 |
| 请求文件太多 | 多个 CSS、JS 和第三方库分别请求，加载链路很长 |
| 浏览器缓存干扰 | 修改 CSS 后刷新，浏览器可能仍使用旧文件 |

构建工具夹在工程、开发者和浏览器之间，把源码转换成更适合开发和上线的形态：

- 在本地启动开发服务器，保存后自动更新页面（热更新）。
- 把多个 CSS、JavaScript 文件合并或优化，减少请求数量。
- 为产物文件名加入内容指纹（hash），内容改变时自动绕过旧缓存。

本节选择的构建工具是 **Vite**。它的名字读作 /viːt/，意思是“快”。

## Node.js：在浏览器外运行 JavaScript

Vite 本身是用 JavaScript 编写的命令行工具，但它不是在浏览器页面中运行，而是在电脑终端中运行。因此需要先安装 Node.js。

Node.js 让 JavaScript 跳出浏览器，在本机直接运行。安装 Node 时还会一并安装 **npm**，它是 JavaScript 项目的包管理工具。

### 按系统安装 Node.js

```bash

# 导入 node 22版本的软件源，且自动 update
curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash -

sudo apt install nodejs npm -y

# 验证版本
node -v
npm -v
```

两条命令都能输出版本号（例如 `v20.x.x` 和 `10.x.x`），说明 Node.js 与 npm 已经准备好。npm 不需要单独安装。

## npm 初始化

进入项目的根目录：

```bash
cd ~/zero-to-tech-4-1
npm init -y
```

`npm init -y` 只做一件事：生成 `package.json`。

`package.json` 是项目的 npm 档案，保存项目名称、版本、依赖和可运行命令。JSON 是一种用键和值表示数据的纯文本格式：

```json
{
  "name": "zero-to-tech",
  "version": "1.0.0"
}
```

对于当前项目，先保留 `name` 和 `version` 即可。`description`、`main`、`keywords`、`author`、`license` 等初始化字段暂时用不上，可以删除。`type` 是否存在取决于 npm 版本和初始化过程，本节不依赖它。

## 安装 Vite

在项目根目录执行：

```bash
npm install -D vite
```

`-D` 是 `--save-dev` 的简写，表示把 Vite 安装为**开发依赖**。Vite 是开发和构建阶段使用的工具，网站运行时不需要把它当作业务库加载。

安装完成后会发生三件事：

1. 创建 `node_modules/`，保存 Vite 及其所有间接依赖。
2. 在 `package.json` 的 `devDependencies` 中记录直接安装的 Vite。
3. 创建或更新 `package-lock.json`，锁定本次实际安装的精确版本。

`package.json`示意：

```json
{
  "name": "zero-to-tech",
  "version": "1.0.0",
  "devDependencies": {
    "vite": "^7.0.0"
  }
}
```

`^7.0.0` 表示允许 7.x 范围内的更新，但不会自动跳到 8.x。`package-lock.json` 则进一步锁定每一个实际安装的版本，应该提交到 Git。

## Vite dev：本地开发与热更新

先在项目目录直接运行安装好的 Vite（ 开发预览 ）：

```bash
./node_modules/.bin/vite
```

Vite 会启动本地开发服务器，通常提供地址：

```text
http://localhost:5173
```

`localhost` 代表当前电脑，`5173` 是本地开发服务器监听的端口。这个请求不会离开本机。

浏览器打开地址后修改源代码并保存，浏览器会自动更新，这叫**热更新（Hot Module Replacement）**。

## Vite build：生成上线产物

开发服务器解决“改完马上看”，上线前还需要把源代码构建成浏览器适合加载的文件。执行：

```bash
./node_modules/.bin/vite build
```

完成后会生成 `dist/`：

```text
dist/
├── index.html
└── assets/
    ├── main-xxxxxxxx.css
    └── main-yyyyyyyy.js
```

构建工具会把多个 CSS、JS 模块合并或优化，并在文件名中加入内容 hash：

| 构建结果 | 解决的问题 |
|------|------|
| 多个 CSS 合并 | 减少浏览器请求次数 |
| 多个 JS 模块合并 | 减少加载和解析开销 |
| 文件名带 hash | 内容改变时自动绕过旧缓存 |

这里可能发现 `dist/` 只有 `index.html`，没有 `text-lab.html`。Vite 默认把根目录 `index.html` 当作入口，多页面需要额外配置。本课程暂不处理，后续 React 会把多个页面合并到一个入口中。

## Vite preview：预览构建结果

`dist/` 不能直接双击打开，因为里面仍可能包含 ES 模块，`file://` 不能提供模块所需的网络来源。使用 Vite 的预览服务器：

```bash
./node_modules/.bin/vite preview
```

它会启动一个服务器来提供 `dist/`，然后在浏览器中检查真实上线产物。打开浏览器的 Network 面板(右键-检查)，可以看到请求数量已经比原始多文件版本少。

`preview` 读取的是 `dist/`，不是源码目录。因此它反映的是 build 后的实际结果，也会暴露多入口配置尚未处理的问题。

## 用 npm run 给命令起短名字

每次输入 `./node_modules/.bin/vite` 太长，可以在 `package.json` 中登记脚本：

```json
{
  "name": "zero-to-tech",
  "version": "1.0.0",
  "scripts": {
    "dev": "vite",
    "build": "vite build",
    "preview": "vite preview"
  }
}
```

以后使用：

```bash
npm run dev
npm run build
npm run preview
```

npm 会读取 `package.json` 的 `scripts`，并自动把 `node_modules/.bin` 加入命令查找路径，所以脚本中的 `vite` 能找到本地安装的 Vite。

`dev`、`build`、`preview` 不是固定关键字.

## dependencies 与 devDependencies

npm 安装的包分两类：

| 类型 | 用途 | 例子 | 安装方式 |
|------|------|------|------|
| `devDependencies` | 只在开发、测试或构建时使用的工具 | Vite | `npm install -D vite` |
| `dependencies` | 网站运行时真正需要的库 | Anime.js | `npm install animejs` |

可以把 Vite 理解成“施工脚手架”，网站建好后不属于房子；把 Anime.js 理解成网站运行时要用的“砖”。因此两者记录位置不同。

## 用 npm 管理 Anime.js

4.1 中的 `cards.js` 和 `score.js` 还从 CDN 导入 Anime.js：

```javascript
import { animate, stagger } from
  "https://cdn.jsdelivr.net/npm/animejs@4/+esm";
```

这种方式能工作，但页面每次都依赖外部 CDN，网络不可用或 URL 行为变化时会影响动画。现在使用 npm 把 Anime.js 安装到项目：

```bash
npm install animejs
```

这次不加 `-D`，因此 Animejs 会被记录到 `dependencies`。安装后，把 `cards.js` 和 `score.js` 顶部的远程 URL 改成包名：

```javascript
import { animate, stagger } from "animejs";
```

浏览器原本不认这个名字，但是 Vite 会在构建时把它解析到 `node_modules` 中的真实文件，并把需要的代码打进最终产物。

重新构建：

```bash
npm run build
```

## 更新 .gitignore

`node_modules/` 和 `dist/` 都是可重建产物，不应提交到 Git。项目根目录的 `.gitignore` 至少写入：

```gitignore
node_modules/
dist/
```

别人拿到这些源头后，可以执行：

```bash
npm install
npm run build
```

重新安装依赖并生成与当前项目一致的构建产物。